# 💼 The Analyst's Notebook · Part 8
### A label for the risk report

Part 6 ended with every column the desk could offer, a ridge penalty chosen on the folds, and a verdict that differed on Apple and on the desk: the wide model beat the single column on 7 of the 11 instruments, and Apple was not one of them.

Part 8 changes the question rather than the columns. The desk does not act on a volatility forecast directly; it acts on whether next month will be busier than this one, because that is when protection has to be bought. That is a label, 1 or 0, and this part fits the first classifier to it: a straight line, which fails; logistic regression on the raw decimal column, which fails in a way that Part 6 prepared you for; and logistic regression on a standardised column, which works. Then the four counts, the AUC, the folds, the 20 columns with `C`, and the desk.

## How to work through this

- Run the **quick load** cell first. It brings back what Part 6 established and loads the price table.
- Each question builds on the last, so keep them in order and keep your variables. Later questions use the names earlier ones created.
- Cells with `...` are blanks. The notebook runs cleanly even before you fill them in, so **Run all** is always safe.
- Hints and solutions are folded under each question. Work first, then check.

**A note on units.** The lecture worked in percent. This notebook keeps the plain decimals of Parts 1 to 6. The label itself has no units, since a comparison is the same in any units, but `LogisticRegression` has a penalty on by default, and a penalty always has units. Q4 and Q5 are about that.

*Stuck for more than 15 minutes? Ask a friend, ask an AI for a hint (not the answer), or email me at `jobo@econ.au.dk`.*

---

## ⚙️ Quick load

The packages, the price table, and what Part 6 left you. Run it and read what it prints.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (mean_squared_error, accuracy_score, confusion_matrix,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import cross_val_score, TimeSeriesSplit, GridSearchCV

CANDIDATE_DIRS = ["data", os.path.join("..", "data"), "."]
REPO_RAW_URL = "https://raw.githubusercontent.com/theill95/mlfin-2026/main/data/"   # used when the CSV files are not next to the notebook


def data_path(filename):
    """Where the course CSV files are, wherever you happen to be running."""
    for folder in CANDIDATE_DIRS:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    if REPO_RAW_URL is not None:
        return REPO_RAW_URL + filename
    raise FileNotFoundError(
        f"Could not find {filename}. Run this notebook from the course folder, "
        f"upload the CSV into Colab, or set REPO_RAW_URL."
    )


def rmse(actual, predicted):
    """Root mean squared error, as in Part 6, as a plain number."""
    return float(np.sqrt(mean_squared_error(actual, predicted)))


# The whole universe: eleven instruments, 2015 to 2024, returns in plain decimals
prices = pd.read_csv(data_path("prices.csv"), parse_dates=["date"])
wide = prices.pivot(index="date", columns="ticker", values="close")
rets = wide.pct_change()
TICKERS = sorted(prices["ticker"].unique())

folds = TimeSeriesSplit(n_splits=5)

# --- What Part 6 established ---
part6_target = "sd of daily returns over the next 20 trading days"
part6_columns = 20                                   # Apple's own windows, up_20d, the other ten instruments' volatility
part6_split = "by date: train to 2022-12-31, test from 2023-01-01"
part6_one_rmse = 0.00412          # one column, vol_20d, on the test block
part6_ridge_rmse = 0.00460        # twenty columns, ridge with alpha chosen on the folds
part6_alpha = 1000                   # the alpha the folds chose
part6_desk_wins = 7                 # instruments where the wide model beat one column, of 11

print("Loaded prices:", prices.shape[0], "rows")
print("Instruments  :", ", ".join(TICKERS))
print()
print("Part 6 left you a wide model and a verdict:")
print("  target  :", part6_target)
print("  columns :", part6_columns, "with alpha", part6_alpha, "chosen on the folds")
print("  split   :", part6_split)
print(f"  test RMSE {part6_ridge_rmse:.5f} for the wide model, {part6_one_rmse:.5f} for one column")
print(f"  the wide model wins on {part6_desk_wins} of {len(TICKERS)} instruments, and not on Apple")
print()
print("Today the target becomes a label.")

---

### Q1 · Where Part 6 stopped

Rebuild Part 6's wide table for Apple as `table`: the six volatility windows `[5, 10, 20, 40, 60, 120]` as `vol_<w>d`, the three return windows `[5, 20, 60]` as `ret_<w>d`, `up_20d`, the 20-day volatility of every **other** instrument as `<ticker>_vol`, the target `vol_next`, and incomplete rows dropped. Split it at the end of 2022 into `train` and `test`, store the twenty feature names in `columns`, refit the one-column regression and check its test RMSE matches `part6_one_rmse`.

In [ ]:
table = pd.DataFrame()

for w in [5, 10, 20, 40, 60, 120]:
    ...

for w in [5, 20, 60]:
    ...

table['up_20d'] = ...

for t in TICKERS:
    ...

table['vol_next'] = ...
table = ...

train = ...
test = ...
columns = ...

one_model = LinearRegression()
...
check = ...
print(check)
print('matches Part 6:', ...)

<details>
<summary>💡 Hint 1</summary>

Column names are text built from the number: `'vol_' + str(w) + 'd'`. Inside the last loop, `if t != 'AAPL':`. `up_20d` is `(rets['AAPL'] > 0).rolling(20).mean()`.

</details>

<details>
<summary>💡 Hint 2</summary>

`columns = list(table.columns[:-1])`. Fit on `train[['vol_20d']]`, score on `test`, and `abs(check - part6_one_rmse) < 0.00001` is the check.

</details>

<details>
<summary>✅ Solution</summary>

```python
table = pd.DataFrame()

for w in [5, 10, 20, 40, 60, 120]:
    table['vol_' + str(w) + 'd'] = rets['AAPL'].rolling(w).std()

for w in [5, 20, 60]:
    table['ret_' + str(w) + 'd'] = rets['AAPL'].rolling(w).mean()

table['up_20d'] = (rets['AAPL'] > 0).rolling(20).mean()

for t in TICKERS:
    if t != 'AAPL':
        table[t + '_vol'] = rets[t].rolling(20).std()

table['vol_next'] = rets['AAPL'].rolling(20).std().shift(-20)
table = table.dropna()

train = table.loc[:'2022-12-31']
test = table.loc['2023-01-01':]
columns = list(table.columns[:-1])

one_model = LinearRegression()
one_model.fit(train[['vol_20d']], train['vol_next'])
check = rmse(test['vol_next'], one_model.predict(test[['vol_20d']]))
print(check)
print('matches Part 6:', abs(check - part6_one_rmse) < 0.00001)
```

0.00412, and `True`: 2,376 rows, 1,894 to fit on and 482 to check on, the same as Part 6. The regression stays in the notebook because the label is made from the same two columns it used.

</details>

---

### Q2 · The label

Add `rising` to `table`: 1 where `vol_next` is larger than `vol_20d`, 0 otherwise. Split again so that `train` and `test` carry the label, print the share of days with a rise in each, and store the accuracy of predicting 0 on every test day as `majority`.

In [ ]:
# add the label to table, then split again
...
train = ...
test = ...

print('train:', ...)
print('test :', ...)
majority = ...
print('majority rule:', majority)

<details>
<summary>💡 Hint 1</summary>

`(table['vol_next'] > table['vol_20d']).astype(int)`: the comparison answers `True` or `False` on every row, and `.astype(int)` makes it 1 or 0.

</details>

<details>
<summary>💡 Hint 2</summary>

The share is the mean of the label. Predicting 0 everywhere is right on the days whose label is 0, so `majority = 1 - test['rising'].mean()`; write it as `max(share, 1 - share)` if you want the rule that also works when the 1s are the majority.

</details>

<details>
<summary>✅ Solution</summary>

```python
table['rising'] = (table['vol_next'] > table['vol_20d']).astype(int)
train = table.loc[:'2022-12-31']
test = table.loc['2023-01-01':]

print('train:', round(train['rising'].mean(), 4))
print('test :', round(test['rising'].mean(), 4))
majority = max(test['rising'].mean(), 1 - test['rising'].mean())
print('majority rule:', round(majority, 4))
```

0.505 of the training days and 0.454 of the test days are followed by a rise, so predicting 0 everywhere scores 0.546 on the test rows. The label is unit-free: multiplying both columns by 100 changes nothing about which is larger. Every model from here has to beat `majority`.

</details>

---

### Q3 · A straight line on the label

Fit `LinearRegression` on `vol_20d` with `rising` as the target as `line`. Count the training days whose fitted value falls below 0 or above 1, and print the fitted value for the most volatile training day.

In [ ]:
line = LinearRegression()
...
fitted = ...

print('outside [0, 1]:', ...)
print('most volatile day:', ...)

<details>
<summary>💡 Hint 1</summary>

`fitted = line.predict(train[['vol_20d']])`; the count is `((fitted < 0) | (fitted > 1)).sum()`.

</details>

<details>
<summary>💡 Hint 2</summary>

`train['vol_20d'].max()` is the largest volatility; predict at it with a one-row DataFrame, `pd.DataFrame({'vol_20d': [train['vol_20d'].max()]})`; `[0]` takes the single number out of the array `predict` returns.

</details>

<details>
<summary>✅ Solution</summary>

```python
line = LinearRegression()
line.fit(train[['vol_20d']], train['rising'])
fitted = line.predict(train[['vol_20d']])

print('outside [0, 1]:', ((fitted < 0) | (fitted > 1)).sum())
print('most volatile day:', line.predict(pd.DataFrame({'vol_20d': [train['vol_20d'].max()]}))[0])
```

28 training days get a fitted value below zero, and the most volatile day, 2020-03-27, at a daily volatility of 0.068, gets -0.60. Read as a probability, that is nonsense, and the line has no way to stop at the edge of the band.

</details>

---

### Q4 · Logistic regression on the raw column

Fit `LogisticRegression()` on `vol_20d` as `raw_model`. Print its intercept and coefficient, the volatility at which its probability crosses one half, the share of test days on which it predicts a rise, and its test accuracy next to `majority`. Something is wrong; say what.

In [ ]:
raw_model = LogisticRegression()
...

print(..., ...)
print('crosses one half at:', ...)
raw_predicted = ...
print('share predicted as a rise:', ...)
print('accuracy:', ..., ' majority:', majority)

<details>
<summary>💡 Hint 1</summary>

The crossing is `-raw_model.intercept_[0] / raw_model.coef_[0, 0]`.

</details>

<details>
<summary>💡 Hint 2</summary>

`raw_predicted.mean()` is the share of ones; `accuracy_score(test['rising'], raw_predicted)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
raw_model = LogisticRegression()
raw_model.fit(train[['vol_20d']], train['rising'])

print(raw_model.intercept_, raw_model.coef_)
print('crosses one half at:', -raw_model.intercept_[0] / raw_model.coef_[0, 0])
raw_predicted = raw_model.predict(test[['vol_20d']])
print('share predicted as a rise:', raw_predicted.mean())
print('accuracy:', accuracy_score(test['rising'], raw_predicted), ' majority:', round(majority, 4))
```

A coefficient of -2.78, a crossing at a volatility of 0.0240, which no test day reaches, so the model predicts a rise on **every** test day: 100% of them. Its accuracy is 0.454, which is one minus `majority`, the share of days with a rise. Every test probability lies between 0.501 and 0.512. The model has barely moved off the intercept.

</details>

---

### Q5 · Why: the penalty has units

`LogisticRegression` has a ridge penalty on by default, at `C=1`. Fit the same model with the penalty switched off in effect, `C=1000000`, as `free_model`, and print its coefficient, its crossing and its test accuracy. Then explain the difference from Q4 in two sentences, using what Part 6 found about alpha on decimal columns.

In [ ]:
free_model = LogisticRegression(C=1000000)
...

print(..., ...)
print('crosses one half at:', ...)
print('accuracy:', ...)

<details>
<summary>💡 Hint 1</summary>

A large `C` is a weak penalty: `C = 1/alpha`.

</details>

<details>
<summary>💡 Hint 2</summary>

In decimals the column's values are around 0.01, a hundred times smaller than in percent, so the coefficient has to be a hundred times larger for the same curve, and the penalty charges its square: ten thousand times more.

</details>

<details>
<summary>✅ Solution</summary>

```python
free_model = LogisticRegression(C=1000000)
free_model.fit(train[['vol_20d']], train['rising'])

print(free_model.intercept_, free_model.coef_)
print('crosses one half at:', -free_model.intercept_[0] / free_model.coef_[0, 0])
print('accuracy:', accuracy_score(test['rising'], free_model.predict(test[['vol_20d']])))
```

A coefficient of -129.7, a crossing at 0.0168 (a daily volatility of 1.68 percent), and an accuracy of 0.571, above `majority`. In percent the coefficient would be -1.29; in decimals it has to be -130, and the default penalty charges for -130 squared, so at `C=1` it shrinks the coefficient to -2.8 and the curve flattens onto the intercept. This is Q6 of Part 6 again: the label has no units, the penalty does.

</details>

---

### Q6 · Standardise, then classify

Build a `Pipeline` with a `StandardScaler` step `'scale'` and a `LogisticRegression()` step `'logit'`, fit it on `vol_20d` as `one_pipe`, and store the test probabilities of a rise as `p_one`. Print the coefficient per standard deviation, the test AUC as `auc_one`, and the test accuracy as `acc_one`. Then print the AUC of `raw_model` from Q4 as well.

In [ ]:
one_pipe = Pipeline([...])
...
p_one = ...

print('per sd:', ...)
auc_one = ...
acc_one = ...
print('AUC:', auc_one, ' accuracy:', acc_one)
print('AUC of the raw model:', ...)

<details>
<summary>💡 Hint 1</summary>

`one_pipe.named_steps['logit'].coef_[0, 0]` is the coefficient; `one_pipe.predict_proba(test[['vol_20d']])[:, 1]` the probabilities.

</details>

<details>
<summary>💡 Hint 2</summary>

`roc_auc_score(test['rising'], p_one)` and `accuracy_score(test['rising'], one_pipe.predict(test[['vol_20d']]))`. The raw model's AUC needs its own `predict_proba`.

</details>

<details>
<summary>✅ Solution</summary>

```python
one_pipe = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
one_pipe.fit(train[['vol_20d']], train['rising'])
p_one = one_pipe.predict_proba(test[['vol_20d']])[:, 1]

print('per sd:', one_pipe.named_steps['logit'].coef_[0, 0])
auc_one = roc_auc_score(test['rising'], p_one)
acc_one = accuracy_score(test['rising'], one_pipe.predict(test[['vol_20d']]))
print('AUC:', auc_one, ' accuracy:', acc_one)
print('AUC of the raw model:', roc_auc_score(test['rising'], raw_model.predict_proba(test[['vol_20d']])[:, 1]))
```

A coefficient of -1.08 per standard deviation, an AUC of 0.797 and an accuracy of 0.571, above `majority`. The raw model's AUC is 0.797, the same number. The AUC only asks whether the days are ranked in the right order, and a coefficient of −2.8 ranks them exactly as −130 does; it is the threshold at one half that the crushed coefficient broke. Standardising inside the pipeline is the fix, as it was for ridge.

</details>

---

### Q7 · The four kinds of day

Turn `p_one` into predictions at one half, build the masks `rose` and `pred_rise`, and count the four kinds of test day. Then print `confusion_matrix` as `cm`, and the precision and recall. For a desk that buys protection on every predicted rise, say in one sentence what the two rates mean.

In [ ]:
predicted = ...
rose = ...
pred_rise = ...

print(..., ..., ..., ...)
cm = ...
print(cm)
print('precision:', ...)
print('recall   :', ...)

<details>
<summary>💡 Hint 1</summary>

`predicted = (p_one >= 0.5).astype(int)`, `rose = test['rising'].values == 1`, `pred_rise = predicted == 1`. The four counts are `(pred_rise & rose).sum()`, `(pred_rise & ~rose).sum()`, `(~pred_rise & rose).sum()` and `(~pred_rise & ~rose).sum()`.

</details>

<details>
<summary>💡 Hint 2</summary>

`confusion_matrix(test['rising'], predicted)`, then `precision_score` and `recall_score` with the true labels first.

</details>

<details>
<summary>✅ Solution</summary>

```python
predicted = (p_one >= 0.5).astype(int)
rose = test['rising'].values == 1
pred_rise = predicted == 1

print((pred_rise & rose).sum(), (pred_rise & ~rose).sum(), (~pred_rise & rose).sum(), (~pred_rise & ~rose).sum())
cm = confusion_matrix(test['rising'], predicted)
print(cm)
print('precision:', precision_score(test['rising'], predicted))
print('recall   :', recall_score(test['rising'], predicted))
```

211 predicted rises that came, 199 that did not, 8 rises missed and 64 calm days predicted as calm. The model predicts a rise on 85% of the test days: recall 0.96, precision 0.51. Protection bought on every predicted rise would have been in place for 96% of the rises, and unnecessary on 49% of the days it was bought. On Apple the crossing point sits above almost the whole of 2023 and 2024, so at one half the model says "rise" nearly always.

</details>

---

### Q8 · The threshold

Loop over the thresholds `[0.4, 0.5, 0.6, 0.7]`: turn `p_one` into predictions at each, and store the precision, the recall and the accuracy as a tuple in a dictionary `by_threshold`. Print it. Which threshold has the highest accuracy, and why is that not yet a choice?

In [ ]:
by_threshold = {}

for threshold in [0.4, 0.5, 0.6, 0.7]:
    ...

for threshold in by_threshold:
    print(threshold, by_threshold[threshold])

<details>
<summary>💡 Hint 1</summary>

Inside the loop, `pred_t = (p_one >= threshold).astype(int)` and the three functions on it, each wrapped in `round(float(...), 3)` and put in a tuple: `(precision, recall, accuracy)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`precision_score(..., zero_division=0)` keeps a threshold that predicts no rise at all from printing a warning.

</details>

<details>
<summary>✅ Solution</summary>

```python
by_threshold = {}

for threshold in [0.4, 0.5, 0.6, 0.7]:
    pred_t = (p_one >= threshold).astype(int)
    by_threshold[threshold] = (round(float(precision_score(test['rising'], pred_t, zero_division=0)), 3),
                               round(float(recall_score(test['rising'], pred_t)), 3),
                               round(float(accuracy_score(test['rising'], pred_t)), 3))

for threshold in by_threshold:
    print(threshold, by_threshold[threshold])
```

At 0.5 the accuracy is 0.571; at 0.6 it is 0.716, with precision 0.67 and recall 0.74. The test rows say a higher threshold would have done better on Apple in 2023 and 2024, and that is exactly the kind of thing the test rows are not allowed to decide. The threshold is a setting, and the next part chooses it from the cost of each error, on the training rows.

</details>

---

### Q9 · The AUC on the folds

Cross-validate the one-column pipeline (a fresh one, unfitted) on the training rows with `folds` and `scoring='roc_auc'` as `auc_folds`. Print the five scores and their mean next to `auc_one`.

In [ ]:
auc_folds = ...
print(...)
print('folds:', ..., ' test:', auc_one)

<details>
<summary>💡 Hint</summary>

`cross_val_score(Pipeline([...]), train[['vol_20d']], train['rising'], cv=folds, scoring='roc_auc')`. Larger is better, so no minus sign.

</details>

<details>
<summary>✅ Solution</summary>

```python
auc_folds = cross_val_score(Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())]),
                            train[['vol_20d']], train['rising'], cv=folds, scoring='roc_auc')
print(auc_folds.round(3))
print('folds:', round(auc_folds.mean(), 4), ' test:', round(auc_one, 4))
```

A mean of 0.779 on the folds, from 0.66 to 0.89, and 0.797 on the test rows. The fold spread is the size of the uncertainty around any one of these numbers.

</details>

---

### Q10 · Twenty columns, and C

Cross-validate a scaling pipeline with `LogisticRegression(max_iter=1000)` on all twenty `columns` and print the mean AUC next to the one-column mean from Q9. Then search `{'logit__C': [0.0001, 0.001, 0.01, 0.1, 1, 10]}` with `GridSearchCV` as `search`, print the best C and score, and store the test AUC of the search as `auc_wide`.

In [ ]:
wide_pipe = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression(max_iter=1000))])
wide_scores = ...
print('twenty columns:', ..., ' one column:', ...)

grid = ...
search = ...
...
print(...)

auc_wide = ...
print('test AUC, twenty columns:', auc_wide, ' one column:', auc_one)

<details>
<summary>💡 Hint 1</summary>

`cross_val_score(wide_pipe, train[columns], train['rising'], cv=folds, scoring='roc_auc')`.

</details>

<details>
<summary>💡 Hint 2</summary>

`GridSearchCV(wide_pipe, grid, cv=folds, scoring='roc_auc')`, then `search.predict_proba(test[columns])[:, 1]` for the test AUC.

</details>

<details>
<summary>✅ Solution</summary>

```python
wide_pipe = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression(max_iter=1000))])
wide_scores = cross_val_score(wide_pipe, train[columns], train['rising'], cv=folds, scoring='roc_auc')
print('twenty columns:', round(wide_scores.mean(), 4), ' one column:', round(auc_folds.mean(), 4))

grid = {'logit__C': [0.0001, 0.001, 0.01, 0.1, 1, 10]}
search = GridSearchCV(wide_pipe, grid, cv=folds, scoring='roc_auc')
search.fit(train[columns], train['rising'])
print(search.best_params_, search.best_score_)

auc_wide = roc_auc_score(test['rising'], search.predict_proba(test[columns])[:, 1])
print('test AUC, twenty columns:', auc_wide, ' one column:', auc_one)
```

Twenty columns score 0.730 on the folds against 0.779 for one. The grid picks C = 0.0001 at 0.745, and on the test rows the wide model scores 0.734 against 0.797 for one column. The verdict of Part 6 holds for the label: on Apple the extra columns do not help, penalty or no penalty.

</details>

---

### Q11 · Which columns the l1 penalty keeps

Fit a scaling pipeline with `LogisticRegression(penalty='l1', solver='liblinear', C=0.01)` on all twenty columns, and store the names of the columns with a non-zero coefficient as the list `kept`. Print it and the test AUC.

In [ ]:
sparse = Pipeline([...])
...

kept = []
...
print(kept)
print('test AUC:', ...)

<details>
<summary>💡 Hint 1</summary>

`sparse.named_steps['logit'].coef_[0]` is the one row of coefficients; loop over `zip(columns, coefs)` and append the name when `b != 0`.

</details>

<details>
<summary>💡 Hint 2</summary>

`roc_auc_score(test['rising'], sparse.predict_proba(test[columns])[:, 1])`.

</details>

<details>
<summary>✅ Solution</summary>

```python
sparse = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression(penalty='l1', solver='liblinear', C=0.01))])
sparse.fit(train[columns], train['rising'])

kept = []
for name, b in zip(columns, sparse.named_steps['logit'].coef_[0]):
    if b != 0:
        kept.append(name)
print(kept)
print('test AUC:', roc_auc_score(test['rising'], sparse.predict_proba(test[columns])[:, 1]))
```

`vol_20d`, `ret_5d`, and an AUC of 0.798. Of twenty columns the penalty keeps the one the lecture started with and the last week's return, which is the same pair it kept on the index. The l1 penalty on a classifier reads like the lasso on a regression.

</details>

---

### Q12 · Across the desk

Write `classifier_vs_majority(ticker)`: build the two columns `vol_20d` and `vol_next` for that ticker from `rets`, drop incomplete rows, add the label, split at the end of 2022, fit the one-column scaling pipeline, and return the triple `(auc, accuracy, majority)` on the test rows. Run it for every ticker into a dictionary `desk`, and count the instruments on which the accuracy beats the majority rule.

In [ ]:
def classifier_vs_majority(ticker):
    ...

desk = {}
for ticker in TICKERS:
    ...

beats = ...
print('beats the majority rule on', beats, 'of', len(desk))

<details>
<summary>💡 Hint 1</summary>

Inside: `now = rets[ticker].rolling(20).std()`, then `frame = pd.DataFrame({'vol_20d': now, 'vol_next': now.shift(-20)}).dropna()`, the label, the split, and the pipeline from Q6 with `tr` and `te`.

</details>

<details>
<summary>💡 Hint 2</summary>

`sum(1 for t in desk if desk[t][1] > desk[t][2])` counts the instruments whose accuracy (second entry) beats the majority rule (third entry).

</details>

<details>
<summary>✅ Solution</summary>

```python
def classifier_vs_majority(ticker):
    now = rets[ticker].rolling(20).std()
    frame = pd.DataFrame({'vol_20d': now, 'vol_next': now.shift(-20)}).dropna()
    frame['rising'] = (frame['vol_next'] > frame['vol_20d']).astype(int)
    tr = frame.loc[:'2022-12-31']
    te = frame.loc['2023-01-01':]

    pipe_t = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
    pipe_t.fit(tr[['vol_20d']], tr['rising'])
    auc = roc_auc_score(te['rising'], pipe_t.predict_proba(te[['vol_20d']])[:, 1])
    accuracy = accuracy_score(te['rising'], pipe_t.predict(te[['vol_20d']]))
    majority_t = max(te['rising'].mean(), 1 - te['rising'].mean())
    return (round(auc, 3), round(accuracy, 3), round(majority_t, 3))

desk = {}
for ticker in TICKERS:
    desk[ticker] = classifier_vs_majority(ticker)

beats = sum(1 for t in desk if desk[t][1] > desk[t][2])
print('beats the majority rule on', beats, 'of', len(desk))
```

11 of 11. The AUC runs from 0.74 on MSFT to 0.90 on DIS, and Apple, at 0.80, is among the lower ones. Unlike the wide regression of Part 6, the one-column classifier beats its baseline everywhere: whether next month is busier than this one is predictable on every instrument, because volatility mean-reverts on every instrument.

</details>

---

### Q13 · Does 0.6 mean 60 percent

Group the test days by the probability `p_one` gave them: below 0.4, 0.4 to 0.5, 0.5 to 0.6, and 0.6 and above. For each group, count the days and compute the share that actually rose, and store the results as a list of tuples `buckets`. Print them. Do the probabilities mean what they say?

In [ ]:
edges = [(0.0, 0.4), (0.4, 0.5), (0.5, 0.6), (0.6, 1.01)]
buckets = []

for low, high in edges:
    ...

for row in buckets:
    print(row)

<details>
<summary>💡 Hint 1</summary>

`mask = (p_one >= low) & (p_one < high)`; the count is `mask.sum()` and the share that rose is `test['rising'].values[mask].mean()`.

</details>

<details>
<summary>💡 Hint 2</summary>

Append `(low, high, int(mask.sum()), round(float(share), 3))`.

</details>

<details>
<summary>✅ Solution</summary>

```python
edges = [(0.0, 0.4), (0.4, 0.5), (0.5, 0.6), (0.6, 1.01)]
buckets = []

for low, high in edges:
    mask = (p_one >= low) & (p_one < high)
    share = test['rising'].values[mask].mean()
    buckets.append((low, high, int(mask.sum()), round(float(share), 3)))

for row in buckets:
    print(row)
```

Below 0.4, 26 days and 0% rose; between 0.5 and 0.6, 168 days and only 29% rose; at 0.6 and above, 242 days and 67% rose. The ranking is right, the levels are not: on Apple in 2023 and 2024 the probabilities run too high, because the training years were more volatile than the test years and the curve was fitted to them. A probability from a model is a claim to check, not a fact.

</details>

---

### Q14 · Draw it

One figure of the test years: `p_one` as a line, a horizontal line at one half, and a dot at the top of the chart on every day that was followed by a rise. Label the axes and give it a title.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

`ax.plot(test.index, p_one, label='probability of a rise')` and `ax.axhline(0.5, color='grey', linestyle='--')`.

</details>

<details>
<summary>💡 Hint 2</summary>

`rose = test['rising'].values == 1`, then `ax.scatter(test.index[rose], np.full(rose.sum(), 1.0), s=6, label='a rise came')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(test.index, p_one, label='probability of a rise')
ax.axhline(0.5, color='grey', linestyle='--')
rose = test['rising'].values == 1
ax.scatter(test.index[rose], np.full(rose.sum(), 1.0), s=6, color='black', label='a rise came')
ax.set_ylabel('probability')
ax.legend(loc='lower left')
ax.set_title('Apple, 2023 to 2024: the probability of a rise, and the rises', loc='left')
plt.show()
```

The line sits above one half for most of the two years, and the dots come in stretches: rises cluster, as calm does. The stretches where the line is high and the dots are absent are the false alarms of Q7, drawn in time.

</details>

---

### Q15 · Write down what you would defend

Finish the way Parts 5 and 6 finished: one dictionary and a function that prints it with a verdict. Fill in `report`, then write `summarise(report)`: it prints each entry on its own line and ends with one sentence on whether the classifier beats the majority rule **on Apple**, and one on the desk as a whole.

In [ ]:
report = {
    'label': ...,
    'share_of_rises_test': ...,
    'majority_rule': ...,
    'one_column_auc': ...,
    'one_column_accuracy': ...,
    'wide_auc': ...,
    'C': ...,
    'threshold': ...,
    'desk_beats_majority': ...,
}

def summarise(report):
    ...

summarise(report)

<details>
<summary>💡 Hint 1</summary>

Most values are already in variables: `majority`, `auc_one`, `acc_one`, `auc_wide`, `search.best_params_['logit__C']`, `beats` from Q12. The threshold is `0.5`, with a note that it was not chosen.

</details>

<details>
<summary>💡 Hint 2</summary>

Inside the function, `for key in report:` then `print(f'{key:22} {report[key]}')`. The verdicts are an `if` on `one_column_accuracy > majority_rule` and a sentence with `desk_beats_majority`.

</details>

<details>
<summary>✅ Solution</summary>

```python
report = {
    'label': 'volatility over the next 20 days above the last 20',
    'share_of_rises_test': round(test['rising'].mean(), 3),
    'majority_rule': round(majority, 3),
    'one_column_auc': round(auc_one, 3),
    'one_column_accuracy': round(acc_one, 3),
    'wide_auc': round(auc_wide, 3),
    'C': search.best_params_['logit__C'],
    'threshold': '0.5, the default, not yet chosen',
    'desk_beats_majority': f'{beats} of {len(desk)}',
}

def summarise(report):
    """Print a finished comparison, and say what the classifier is worth."""
    for key in report:
        print(f'{key:22} {report[key]}')

    if report['one_column_accuracy'] > report['majority_rule']:
        print('\nOn Apple, the one-column classifier beats the majority rule; the twenty columns do not add to it.')
    else:
        print('\nOn Apple, the classifier does not beat the majority rule at this threshold.')
    print(f"Across the desk, it beats the majority rule on {report['desk_beats_majority']} instruments.")

summarise(report)
```

Nine lines and two verdicts, and this time they point the same way. The report carries the threshold as an open item on purpose: the model's probabilities rank the days well and sit too high, so what the desk does with a 0.55 is a decision about costs, and that is the next part.

</details>

---

## 🧭 What you have now

| what | where it lives |
|:--|:--|
| Part 6's table and split, rebuilt | `table`, `train`, `test`, `columns`, `one_model` |
| the label and the share to beat | `table['rising']`, `majority` |
| the straight line that fails | `line` |
| logistic regression on the raw column, and without its penalty | `raw_model`, `free_model` |
| the standardised one-column classifier and its probabilities | `one_pipe`, `p_one`, `auc_one`, `acc_one` |
| the four counts | `cm` |
| the thresholds | `by_threshold` |
| the folds | `auc_folds` |
| twenty columns with C chosen | `search`, `auc_wide` |
| the l1 penalty's columns | `kept` |
| the whole desk | `desk`, `beats` |
| the probability check | `buckets` |
| the thing you would defend | `report` |

## What changed since Part 6

- **The target became a label.** Whether next month is busier than this one, 1 or 0, with a rise on 45% of the test days and a majority rule at 0.546.
- **The units problem came back in a new form.** The label has no units, but the default penalty does: on the raw decimal column it shrank the coefficient from -130 to -2.8 and the model predicted a rise on every day (Q4 and Q5). Standardising in a pipeline repaired it (Q6), and the AUC, which only ranks, never noticed.
- **The one-column classifier beats its baseline everywhere.** 0.571 against 0.546 on Apple, and on 11 of 11 instruments across the desk (Q12).
- **The twenty columns still do not help on Apple.** 0.734 against 0.797, with C chosen on the folds (Q10); the l1 penalty keeps `vol_20d` and `ret_5d` (Q11).
- **The probabilities rank well and run high.** In the 0.5 to 0.6 group only 29% of the days rose (Q13), so the threshold at one half predicts a rise far too often on these two years.

## Where this leaves the risk report

The report now answers a yes/no question with a probability, and it can say how well that probability ranks the days: an AUC of about 0.8 on Apple and between 0.74 and 0.90 across the desk. What it cannot yet say is what to do with a probability of 0.55. At one half the model buys protection almost every month; at 0.6 it buys it far less often and misses a quarter of the rises. Which is right depends on what a false alarm costs and what a miss costs.

**Next part:** the threshold as a decision, with the cost of each kind of error, chosen on the training rows; and a label that is rare rather than balanced.